In [1]:
!pip install arch -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.3/981.3 kB 25.8 MB/s eta 0:00:00


In [2]:
import os, sys
import numpy as np
import pandas as pd
from google.colab import drive
DRIVE = "/content/drive/MyDrive/volatility-forecast"
drive.mount("/content/drive", force_remount=True)
os.chdir(DRIVE)

Mounted at /content/drive


In [3]:
from arch import arch_model
from src.conformal import frozen_har_residuals, rolling_sigma

# --- load work frame (3831-row dropna'd frame, same one 03/04/06 used) ---
df = pd.read_csv('data/processed/dataset.csv', index_col=0, parse_dates=True)
df = df.iloc[21:-21]  # head21 + tail21 NaN trim -> 3831-row work frame
assert len(df) == 3831, f"expected 3831, got {len(df)}"

HORIZON = 'y_rv21'  # main horizon (h1 control comes later)
X = df[['rv1', 'rv5', 'rv21']].values
y = df[HORIZON].values
dates = df.index

# --- (1) frozen HAR: fit once on first 1008 rows, forward-predict, no retrain ---
yhat, resid, beta = frozen_har_residuals(X, y, n_train=1008)
print("frozen HAR beta [const, b1, b5, b21]:", np.round(beta, 4))
print("residual stream: mean %.5f  std %.5f" % (resid.mean(), resid.std()))

# --- (2) sigma_GARCH: conditional vol sqrt(h_t) from a GARCH(1,1) on rv1 ---
# arch fits on percent scale (x100); divide output back to RV decimal scale.
am = arch_model(df['rv1'].values * 100, vol='Garch', p=1, q=1, mean='Zero')
garch_res = am.fit(disp='off')
sigma_garch = garch_res.conditional_volatility / 100.0  # back to RV units
print("sigma_GARCH: mean %.5f  (vs resid std %.5f)" % (sigma_garch.mean(), resid.std()))

# --- (3) sigma_roll: trailing rolling std of rv1, k=21, leakage-free (shift inside) ---
sigma_roll = rolling_sigma(df['rv1'].values, k=21)
print("sigma_roll : mean %.5f  (NaN in first 21)" % np.nanmean(sigma_roll))

# --- assemble aligned frame; stream[0] = row 1008 = f00 test start ---
out = pd.DataFrame({
    'date': dates, 'y': y, 'yhat': yhat, 'resid': resid,
    'sigma_garch': sigma_garch, 'sigma_roll': sigma_roll,
}).set_index('date')
print("\nstream head (around row 1008 = f00 test start 2015-02-05):")
print(out.iloc[1006:1010].round(5))

frozen HAR beta [const, b1, b5, b21]: [0.0048 0.0258 0.201  0.2949]
residual stream: mean 0.00088  std 0.00501
sigma_GARCH: mean 0.01193  (vs resid std 0.00501)
sigma_roll : mean 0.00731  (NaN in first 21)

stream head (around row 1008 = f00 test start 2015-02-05):
                  y     yhat    resid  sigma_garch  sigma_roll
date                                                          
2015-02-03  0.00627  0.01021 -0.00394      0.01161     0.00648
2015-02-04  0.00675  0.00979 -0.00305      0.01141     0.00630
2015-02-05  0.00651  0.00988 -0.00337      0.01071     0.00650
2015-02-06  0.00757  0.00967 -0.00210      0.01053     0.00643


In [4]:
# Cell 1: split + normalized conformal, per-fold coverage (alpha=0.1, nominal 90%)
for m in list(sys.modules):
    if m.startswith('src'):
        del sys.modules[m]
from src.conformal import split_conformal_coverage, normalized_conformal_coverage
from src.splits import walk_forward_splits

ALPHA = 0.1
NOMINAL = 1 - ALPHA

# work-frame arrays already aligned (out.index == df.index, stream[0]=row1008)
resid_all = out['resid'].values
sig_g = out['sigma_garch'].values
sig_r = out['sigma_roll'].values

# 06 drift z-scores per fold (locked ground truth) for later money plot
DRIFT_Z = {0:None, 5:0.19, 7:2.11, 8:-0.21}  # fill rest from 06 csv if needed

rows = []
splits = list(walk_forward_splits(len(df), min_train=1008, test_size=252, embargo=21))
for fold, (tr_idx, te_idx) in enumerate(splits):
    # calibration = 252 rows immediately BEFORE this fold's test block (leak-free)
    cal_idx = np.arange(te_idx[0] - 252, te_idx[0])
    # guard: sigma_roll has NaN in first 21 rows; skip if calibration touches them
    if np.isnan(sig_r[cal_idx]).any() or np.isnan(sig_g[cal_idx]).any():
        continue

    rc, rt = resid_all[cal_idx], resid_all[te_idx]
    cov_s, q_s, w_s = split_conformal_coverage(rc, rt, ALPHA)
    cov_g, q_g, w_g = normalized_conformal_coverage(rc, sig_g[cal_idx], rt, sig_g[te_idx], ALPHA)
    cov_r, q_r, w_r = normalized_conformal_coverage(rc, sig_r[cal_idx], rt, sig_r[te_idx], ALPHA)

    rows.append({
        'fold': fold,
        'test_start': dates[te_idx[0]].date(),
        'cov_split': round(cov_s, 3),
        'cov_norm_garch': round(cov_g, 3),
        'cov_norm_roll': round(cov_r, 3),
        'width_split': round(w_s, 5),
        'width_garch': round(w_g, 5),
        'width_roll': round(w_r, 5),
    })

cov_df = pd.DataFrame(rows)
print(f"nominal coverage = {NOMINAL:.0%}\n")
print(cov_df.to_string(index=False))
print(f"\nmean coverage  split={cov_df.cov_split.mean():.3f}  "
      f"garch={cov_df.cov_norm_garch.mean():.3f}  roll={cov_df.cov_norm_roll.mean():.3f}")

nominal coverage = 90%

 fold test_start  cov_split  cov_norm_garch  cov_norm_roll  width_split  width_garch  width_roll
    0 2015-02-05      0.845           0.881          0.909      0.00960      0.01159     0.01501
    1 2016-02-05      0.988           0.917          0.786      0.01342      0.01141     0.01044
    2 2017-02-06      0.921           0.917          0.881      0.01085      0.00926     0.01001
    3 2018-02-06      0.683           0.837          0.897      0.00946      0.01423     0.02063
    4 2019-02-07      0.976           0.881          0.869      0.01718      0.01377     0.01437
    5 2020-02-07      0.730           0.861          0.885      0.01247      0.02842     0.03508
    6 2021-02-08      1.000           0.988          1.000      0.02518      0.02323     0.02444
    7 2022-02-07      0.802           0.921          0.933      0.01458      0.02121     0.02272
    8 2023-02-08      1.000           1.000          0.996      0.01730      0.01242     0.01076
    9 

In [5]:
for m in list(sys.modules):
    if m.startswith('src'):
        del sys.modules[m]
from src.conformal import aci_stream_coverage

# build {fold -> absolute test indices}, same splits as Cell 1
test_blocks = {fold: te_idx for fold, (tr_idx, te_idx) in enumerate(splits)}

fold_cov_aci, alpha_path = aci_stream_coverage(
    resid_all, sig_g, test_blocks,
    cal_window=252, alpha=ALPHA, gamma=0.01,
)

# attach cov_aci to the Cell-1 table for side-by-side comparison
cov_df['cov_aci'] = cov_df['fold'].map(lambda f: round(fold_cov_aci[f], 3))

cols = ['fold', 'test_start', 'cov_split', 'cov_norm_garch', 'cov_aci']
print(f"nominal coverage = {NOMINAL:.0%}  (gamma=0.01)\n")
print(cov_df[cols].to_string(index=False))
print(f"\nmean  split={cov_df.cov_split.mean():.3f}  "
      f"garch={cov_df.cov_norm_garch.mean():.3f}  "
      f"aci={cov_df.cov_aci.mean():.3f}")
print(f"alpha_t range over stream: [{alpha_path.min():.3f}, {alpha_path.max():.3f}]")

nominal coverage = 90%  (gamma=0.01)

 fold test_start  cov_split  cov_norm_garch  cov_aci
    0 2015-02-05      0.845           0.881    0.897
    1 2016-02-05      0.988           0.917    0.913
    2 2017-02-06      0.921           0.917    0.865
    3 2018-02-06      0.683           0.837    0.925
    4 2019-02-07      0.976           0.881    0.885
    5 2020-02-07      0.730           0.861    0.940
    6 2021-02-08      1.000           0.988    0.857
    7 2022-02-07      0.802           0.921    0.937
    8 2023-02-08      1.000           1.000    0.893
    9 2024-02-09      0.647           0.631    0.861
   10 2025-02-12      0.734           0.829    0.909

mean  split=0.848  garch=0.878  aci=0.898
alpha_t range over stream: [0.001, 0.293]


In [12]:
for m in list(sys.modules):
    if m.startswith("src"):
        del sys.modules[m]
from src.conformal import bayes_nig_fit, bayes_predict_interval

ALPHA = 0.10          # nominal 90% intervals
N_CAL = 1008          # same frozen calibration window as 06/07
TARGET = "y_rv21"     # h21 horizon (where the drift signal lives)

# HAR design matrix: intercept + rv1 + rv5 + rv21
def har_design(df):
    X = np.column_stack([
        np.ones(len(df)),
        df["rv1"].values,
        df["rv5"].values,
        df["rv21"].values,
    ])
    return X

X_all = har_design(df)
y_all = df[TARGET].values

# freeze posterior on first N_CAL rows
post = bayes_nig_fit(X_all[:N_CAL], y_all[:N_CAL])

bayes_rows = []
for i, (tr_idx, te_idx) in enumerate(splits):
    Xte, yte = X_all[te_idx], y_all[te_idx]
    lo, hi, yhat = bayes_predict_interval(post, Xte, alpha=ALPHA)
    covered = (yte >= lo) & (yte <= hi)
    bayes_rows.append({
        "fold": i,
        "coverage": covered.mean(),
        "mean_width": (hi - lo).mean(),
    })

bayes_df = pd.DataFrame(bayes_rows)
print(bayes_df.to_string(index=False))
print(f"\nmean coverage: {bayes_df['coverage'].mean():.3f}  (nominal {1-ALPHA:.2f})")



 fold  coverage  mean_width
    0       1.0    0.147322
    1       1.0    0.147222
    2       1.0    0.147192
    3       1.0    0.147512
    4       1.0    0.147215
    5       1.0    0.148641
    6       1.0    0.147286
    7       1.0    0.147885
    8       1.0    0.147194
    9       1.0    0.147255
   10       1.0    0.147705

mean coverage: 1.000  (nominal 0.90)
